# Trabajo final · Data Analyst · Vitalia Gimnasios

**Red de gimnasios con planes mensuales** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿Por qué suben las bajas de socios en el primer trimestre de 2025?**

**Decisión que habilita:** Si hay que cambiar el servicio, la captación o el seguimiento de socios nuevos.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de Vitalia Gimnasios

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNqdV+tu2zYU/p+nIAR0oDZHkGU7bZN6P9aiQH/MHdBiwCAIBiPRCVuJFEjZbZKl77Jn2YvtO6QsX5oUwQJHF/Lcz/nOoVTT"
    "GtsxvW7aGyYc0+2ItUJXeMSvrU4quWJLty6V4XbEnLRKjthGWCV0Jx2I4/MThj/H5mE3KU17w+NEuO6mldxcfpJlF1+wBgQ2"
    "sZBtGl5LzV0cs1esDdx5U2A/HwTzzYjZmK2MZRumtCcoLpiV3dri7STY1YibtVtK1wqyz7Ocb2nyTVKbL9LyGPYm67btH9kv"
    "LGLRiC543BS5TRQ0XknreDpi07gIwiuRLTeqE7US3ElZzbM06321MFW3vS8JiMW67pZWX3lC+LqCxXNEL/moGuk60bQ8Avvs"
    "dJydTsYRKGh/mqWplyfqTjxEPz1Nx/hFMQzFZmeWHfYrCXJ+aPXzFBcN99ZadfPoDaloa6F9zMtro0rJ8+i3f/9xqjTk/B/1"
    "2tH97bquowK8yOQ8T6azEUsmdMnSAjJKoUV9KOSLvCRGBzvobuUKWa/MnhDPP90K8R5S8s9m6QUT2cOBIUezJzg6wX9z6Gcf"
    "PSSkNLoUndT45zktj6Cv2IXimISWR7QqrBU3ewEq2M/Qsh+AY1a/fsBLcdnxtdY05gE+vN9KaxyH4ktj6tjLMBoV3/QrxL4g"
    "VgSjCehAKFyI2xvRibdWNJLfRapa+iycs3wVfbhT95HHiyK8oDavJB+jMmIkJtKmubSeMnotdWd9DSyAfJ/DD2tLt/fIiH9/"
    "bZAaevhT1LWkzEalIIxVoiIZE6q2LKMLPY1f0OWMLtO0uIf517fBdSdrgD8Ems3nbK8ChyVfiVCRJ+nkDDWTZs8Luo1fhtK5"
    "FJ/69K5Qq3zhA1YhmFQiZ1MeLcTHCGGsqN3Mo91Orl1B9bFRmz4TPsyLPswoHKdcR+3uylB0czKi8DopjJ93YcymPfB9KW9T"
    "8UOsIk/y/WrlZMcbo7trN/8c+sIy8NP1EcJxzE638gMEKnFDy4MJouwgwvv1E+O+/l/Nj8LSq4qhVSJtS0Ig51hJbqSwqNNx"
    "RvWFd6+WdPLeHJLwTlfyqxcdHzA8QhJkxMlG1GvpUADQdn3bjwK85gEPsLY35tc5g5+7d9ifxTQCkvHzwU8naoklcheUw/BY"
    "0Ny4jo+lXbBKCd+rhoaBHpS9GLEF9qiMchJISvgQ/oNWA/6e5JSN93rMzi+K+SDlraidHIxV1ddQZV+upZUcRsd5ipFVi2Z/"
    "3QciB3Gxsz+8Agwp6jhLshF7maTxd6yk15MCaAlcw7Yvb+edbo1yzmhOq4NRvh9g6PkZeqtaDvYRsWBO+vJPBAajrjgHFRWD"
    "6+yKAsKjZ3+dPmsIWIgm32yGB1SCTbSxjaj5WUaYj/E3aGwtpYx/66uTwsD+ZvzBHALrse+VEuP7f3QMNB7fic5ww/uE7ke+"
    "k9/w+Tyg/IneBoNyVeA9Ug2xRkytWJ+BgsGH3VnGezKbxQz6+OEq/GMSRcIiyBAYkn2gnIH8Rzo6bYWW/vudOk9n1fdtfeF7"
    "+kqW12JJ+AN5KOUt/ntUHrl3+qwiDyMKJLH46Rf5SbYV04+1vdEyTH4/hJJ+A5gKs57gldAkAG7pMqVLRv37pY+eDCODxmCN"
    "8hsqZzJB5aQETRoddICJE2vWyE18vx+jvHeT4BsVIWIf6JjpOC3FSdU96GYPmW8gV06LQLzt+ceB7wdBaep1o908jxQNPxQI"
    "3QAWnAL9Y6P0ujOova2gfJcxsi2YnGzX+q6RB6CpYqc9rFTWtHyrFDrjfE/ejw0IY2o7uQ6c8YuPOaP8cd8PeZpddGwjABLL"
    "03wJMCJfBuV+5Ym+PKC/6Kd8tX/MXISTnpUo0lLOfaeFob1RtSlzMAwYGIqDH8HgEXpf7MXDYzZL40eL6sCAfVun39m6Lf3i"
    "4PBMR6QZtappeizssC+OZ3sgLEJr1EJfDKjwGKaNvW+zIC7A+vDTiPA4CbDqv47u/AHeAZr+Tgd6z04L/gErvkqlLpVwOEpo"
    "t0b3I4IeLpHPfNS31vuTkyuppRX0cbT37XTyH89vZpY="
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. La baja mensual por trimestre (`socios` y su `fecha_baja`).
2. La baja a 90 días por plan y canal de alta.
3. Un control de integridad: ¿hay fechas de baja anteriores a la fecha de alta?

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`visitas` (visitas por mes)): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (meses de antigüedad, visitas del primer mes, impago previo, cohorte de alta).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: tasa de baja mensual (bajas del mes sobre socios activos al inicio), baja a 90 días por cohorte, visitas por socio e impagos.

**Segmentación:** plan, canal de alta, cohorte de alta y sede.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.